In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
from fraud_detection.config import load_config
from fraud_detection.data.load import load_interim
from fraud_detection.data.split import assign_split
from fraud_detection.evaluation.metrics import evaluate
from fraud_detection.features.preprocessing import TARGET, Preprocessor
from fraud_detection.models.xgb import train_from_config

In [ ]:
data_cfg = load_config("data")
df = load_interim()
split = assign_split(df, **data_cfg["split"])
train, val = df[split == "train"], df[split == "val"]

In [ ]:
prep = Preprocessor(**data_cfg["preprocessing"]).fit(train)
X_train, y_train = prep.transform(train), train[TARGET]
X_val, y_val = prep.transform(val), val[TARGET]

## Pondération des fraudes avec XGB

On indique ici à XGB que chaque fraude compte autant que les transactions légitimes réunies.

In [ ]:
cfg = load_config("xgboost_tuned")

In [ ]:
ratio = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight = {ratio:.1f}")

cfg["name"] = "xgb_weighted"
cfg["params"]["scale_pos_weight"] = float(ratio)
model_w, params_w, _ = train_from_config(X_train, y_train, X_val, y_val, cfg)

scores_w = model_w.predict_proba(X_val)[:, 1]
pd.Series(evaluate(y_val, scores_w)).round(4)